EasyWave Model (CPU):

In [ ]:
import subprocess
import glob
import os
import shutil
from pathlib import Path
import numpy as np
import xarray as xr
import struct

# ================== FUNCIONES AUXILIARES ==================
def leer_grilla_easywave_ascii(grd_file):
    with open(grd_file, 'r') as f:
        assert f.readline().strip() == 'DSAA'
        ncols, nrows = map(int, f.readline().split())
        xmin, xmax = map(float, f.readline().split())
        ymin, ymax = map(float, f.readline().split())
        _ = f.readline()
        z = np.empty((nrows, ncols), dtype=np.float32)
        for r in range(nrows):
            vals = f.readline().split()
            row = np.zeros(ncols, dtype=np.float32)
            for c in range(min(len(vals), ncols)):
                row[c] = float(vals[c])
            z[r, :] = row
    return z, nrows, ncols, xmin, xmax, ymin, ymax


def leer_sshmax_subdom(path):
    path = Path(path)
    with path.open('rb') as f:
        if f.read(4) != b'DSBB':
            raise ValueError(f'{path} no comienza con DSBB')
        hdr = f.read(52)
        nI, nJ, loMin, loMax, laMin, laMax, t0, t1 = struct.unpack('<hh6d', hdr)
        _ = struct.unpack('<2d', f.read(16))
        data = np.fromfile(f, dtype='<f4')
        if data.size < nI*nJ:
            tmp = np.zeros(nI*nJ, dtype=np.float32)
            tmp[:data.size] = data
            data = tmp
        arr = data.reshape((nJ, nI))
    return arr, (nI, nJ), (loMin, loMax, laMin, laMax)


def colocar_subdom_en_full(full_shape, domain_bounds, sub_arr, sub_bounds):
    nrows, ncols = full_shape
    xmin, xmax, ymin, ymax = domain_bounds
    loMin, loMax, laMin, laMax = sub_bounds

    lon_full = np.linspace(xmin, xmax, ncols)
    lat_full = np.linspace(ymin, ymax, nrows)

    i0 = int(np.argmin(np.abs(lon_full - loMin)))
    i1 = int(np.argmin(np.abs(lon_full - loMax))) + 1
    j0 = int(np.argmin(np.abs(lat_full - laMin)))
    j1 = int(np.argmin(np.abs(lat_full - laMax))) + 1

    nJ, nI = sub_arr.shape
    if (j1 - j0) != nJ:
        j1 = j0 + nJ
    if (i1 - i0) != nI:
        i1 = i0 + nI

    full = np.zeros((nrows, ncols), dtype=np.float32)
    full[j0:j1, i0:i1] = sub_arr
    return full, lat_full, lon_full


def generar_netcdf(grd_file, sshmax_file, run_out, nc_file):
    bathy, nrows, ncols, xmin, xmax, ymin, ymax = leer_grilla_easywave_ascii(grd_file)
    sub_arr, (nI, nJ), sub_bounds = leer_sshmax_subdom(sshmax_file)
    max_height_full, lat, lon = colocar_subdom_en_full(
        (nrows, ncols),
        (xmin, xmax, ymin, ymax),
        sub_arr,
        sub_bounds
    )

    eta_list = []
    time_list = []
    ssh_files = sorted(Path(run_out).glob("eWave.2D.*.ssh"))

    for f in ssh_files:
        try:
            t = int(f.stem.split(".")[2])  # tiempo en segundos
        except Exception:
            continue
        arr, (nI, nJ), sub_bounds_t = leer_sshmax_subdom(f)
        eta_full, _, _ = colocar_subdom_en_full(
            (nrows, ncols),
            (xmin, xmax, ymin, ymax),
            arr,
            sub_bounds_t
        )
        eta_list.append(eta_full)
        time_list.append(t)

    ds_vars = {
        "original_bathy": (["lat", "lon"], bathy),
        "deformed_bathy": (["lat", "lon"], bathy),
        "max_height":     (["lat", "lon"], max_height_full),
    }

    if eta_list:
        eta = np.stack(eta_list, axis=0)
        ds_vars["eta"] = (["time", "lat", "lon"], eta)

    ds = xr.Dataset(
        data_vars=ds_vars,
        coords={
            "lat": lat,
            "lon": lon,
            "time": time_list if eta_list else []
        },
        attrs={"source": "EasyWave outputs (sshmax + series eta)"}
    )

    ds.to_netcdf(nc_file)
    print(f"✅ NetCDF listo: {nc_file}")
    if eta_list:
        print(f"   Incluye {len(eta_list)} pasos de tiempo (eta)")
    else:
        print("⚠️ No se encontraron archivos .ssh (solo se incluyó sshmax)")

from datetime import timedelta

def generar_netcdf_eta(ssh_file, output_nc):
    """
    Convierte el archivo ASCII eWave.poi.ssh en un NetCDF con variable 'eta'
    de dimensiones (time, grid_npoints).
    El tiempo se almacena como strings en formato HH:MM:SS.
    """
    print(f"📈 Procesando mareogramas: {ssh_file}")

    # Leemos todas las líneas no vacías
    with open(ssh_file, "r") as f:
        lineas = [line.strip() for line in f if line.strip()]

    # Detectar y eliminar encabezado (líneas no numéricas)
    datos_numericos = []
    for line in lineas:
        partes = line.split()
        try:
            float(partes[0])  # si esto falla, no es una línea numérica
            datos_numericos.append(line)
        except ValueError:
            continue

    if not datos_numericos:
        print("⚠️ No se encontraron datos numéricos en el archivo.")
        return

    # Convertir a matriz numérica
    data = np.array([list(map(float, l.split())) for l in datos_numericos], dtype=np.float32)

    tiempo_min = data[:, 0]
    eta_vals = data[:, 1:]

    # Convertimos el tiempo a formato HH:MM:SS
    tiempo_hms = [
        str(timedelta(seconds=int(t * 60))) for t in tiempo_min
    ]

    # Crear dataset
    ds = xr.Dataset(
        data_vars={
            "eta": (["time", "grid_npoints"], eta_vals)
        },
        coords={
            "time": tiempo_hms,
            "grid_npoints": np.arange(eta_vals.shape[1])
        },
        attrs={
            "description": "Mareogramas simulados por EasyWave",
            "units": "meters",
            "time_format": "HH:MM:SS desde inicio de simulación"
        }
    )

    ds.to_netcdf(output_nc)
    print(f"✅ Archivo creado: {output_nc}")
    print(f"   Dimensiones: time={eta_vals.shape[0]}, grid_npoints={eta_vals.shape[1]}")
    print(f"   Ejemplo tiempo: {tiempo_hms[0]} → {tiempo_hms[-1]}")


# ================== MAIN ==================
grd_file = r"Datos/EasyWave/GrdASCII/grilla_ascii.grd"
flt_base = r"Datos/simulaciones-tsunami-hysea"
out_folder = r"Datos/EasyWave/Outputs"
poi_file = r"Datos/EasyWave/puntos.poi"
os.makedirs(out_folder, exist_ok=True)

flt_files = glob.glob(os.path.join(flt_base, "**", "*.flt"), recursive=True)

for flt_file in flt_files:
    rel_path = os.path.relpath(flt_file, flt_base)
    sim_name = os.path.splitext(rel_path.replace(os.sep, "_"))[0]

    run_out = os.path.join(out_folder, sim_name)
    os.makedirs(run_out, exist_ok=True)

    print(f"🌊 Procesando {sim_name} ...")

    subprocess.run([
        "easywave",
        "-grid", os.path.abspath(grd_file),
        "-source", os.path.abspath(flt_file),
        "-poi", os.path.abspath(poi_file),
        "-time", "120"
    ], cwd=run_out)

    subprocess.run(["sshmax2png.sh", "-grd", os.path.abspath(grd_file)], cwd=run_out)
    subprocess.run(["ssh2png.sh", "-grd", os.path.abspath(grd_file), "03600"], cwd=run_out)

    flt_dir = os.path.dirname(flt_file)
    file_heig = os.path.join(run_out, "eWave.2D.png")
    file_prop = os.path.join(run_out, "eWave.2D.03600.png")
    new_heig = os.path.join(flt_dir, "max_wave_heights.png")
    new_prop = os.path.join(flt_dir, "wave_propagation.png")

    if os.path.exists(file_heig):
        shutil.move(file_heig, new_heig)
    if os.path.exists(file_prop):
        shutil.move(file_prop, new_prop)

    # generar NetCDF con sshmax + series eta
    sshmax_file = os.path.join(run_out, "eWave.2D.sshmax")
    nc_file = os.path.join(flt_dir, "resultado_easywave.nc")
    if os.path.exists(sshmax_file):
        generar_netcdf(grd_file, sshmax_file, run_out, nc_file)

    # === Generar NetCDF con serie temporal eta ===
    ssh_file = os.path.join(run_out, "eWave.poi.ssh")
    nc_eta_file = os.path.join(flt_dir, "resultado_ts_easywave.nc")

    if os.path.exists(ssh_file):
        generar_netcdf_eta(ssh_file, nc_eta_file)
    else:
        print("⚠️ No se encontró el archivo eWave.poi.ssh (no se generó resultado_ts_easywave.nc)")


print("✅ Todos los procesos han finalizado correctamente.")


Procesando mw_8.1_02083_02083 ...

easyWave ver.2013-04-11
Model time = 00:00:00,   elapsed: 226164 msec
Model time = 00:10:00,   elapsed: 226519 msec
Model time = 00:20:00,   elapsed: 227343 msec
Model time = 00:30:00,   elapsed: 229048 msec
Model time = 00:40:00,   elapsed: 231861 msec
Model time = 00:50:00,   elapsed: 235763 msec
Model time = 01:00:00,   elapsed: 240284 msec
Model time = 01:10:00,   elapsed: 245347 msec
Model time = 01:20:00,   elapsed: 251049 msec
Model time = 01:30:00,   elapsed: 257483 msec
Model time = 01:40:00,   elapsed: 264668 msec
Model time = 01:50:00,   elapsed: 272734 msec
Model time = 02:00:00,   elapsed: 281160 msec
✅ NetCDF listo: Datos/simulaciones-tsunami-hysea/mw_8.1/02083/resultado_easywave.nc
Procesando mw_8.3_03529_03529 ...

easyWave ver.2013-04-11
Model time = 00:00:00,   elapsed: 225688 msec
Model time = 00:10:00,   elapsed: 226127 msec
Model time = 00:20:00,   elapsed: 227099 msec
Model time = 00:30:00,   elapsed: 228954 msec
Model time = 00:

EasyWave Model (GPU):

In [ ]:
import subprocess
import glob
import os
import shutil
from pathlib import Path
import numpy as np
import xarray as xr
import struct

# ================== FUNCIONES AUXILIARES ==================
def leer_grilla_easywave_ascii(grd_file):
    with open(grd_file, 'r') as f:
        assert f.readline().strip() == 'DSAA'
        ncols, nrows = map(int, f.readline().split())
        xmin, xmax = map(float, f.readline().split())
        ymin, ymax = map(float, f.readline().split())
        _ = f.readline()
        z = np.empty((nrows, ncols), dtype=np.float32)
        for r in range(nrows):
            vals = f.readline().split()
            row = np.zeros(ncols, dtype=np.float32)
            for c in range(min(len(vals), ncols)):
                row[c] = float(vals[c])
            z[r, :] = row
    return z, nrows, ncols, xmin, xmax, ymin, ymax


def leer_sshmax_subdom(path):
    path = Path(path)
    with path.open('rb') as f:
        if f.read(4) != b'DSBB':
            raise ValueError(f'{path} no comienza con DSBB')
        hdr = f.read(52)
        nI, nJ, loMin, loMax, laMin, laMax, t0, t1 = struct.unpack('<hh6d', hdr)
        _ = struct.unpack('<2d', f.read(16))
        data = np.fromfile(f, dtype='<f4')
        if data.size < nI*nJ:
            tmp = np.zeros(nI*nJ, dtype=np.float32)
            tmp[:data.size] = data
            data = tmp
        arr = data.reshape((nJ, nI))
    return arr, (nI, nJ), (loMin, loMax, laMin, laMax)


def colocar_subdom_en_full(full_shape, domain_bounds, sub_arr, sub_bounds):
    nrows, ncols = full_shape
    xmin, xmax, ymin, ymax = domain_bounds
    loMin, loMax, laMin, laMax = sub_bounds

    lon_full = np.linspace(xmin, xmax, ncols)
    lat_full = np.linspace(ymin, ymax, nrows)

    i0 = int(np.argmin(np.abs(lon_full - loMin)))
    i1 = int(np.argmin(np.abs(lon_full - loMax))) + 1
    j0 = int(np.argmin(np.abs(lat_full - laMin)))
    j1 = int(np.argmin(np.abs(lat_full - laMax))) + 1

    nJ, nI = sub_arr.shape
    if (j1 - j0) != nJ:
        j1 = j0 + nJ
    if (i1 - i0) != nI:
        i1 = i0 + nI

    full = np.zeros((nrows, ncols), dtype=np.float32)
    full[j0:j1, i0:i1] = sub_arr
    return full, lat_full, lon_full


def generar_netcdf(grd_file, sshmax_file, run_out, nc_file):
    bathy, nrows, ncols, xmin, xmax, ymin, ymax = leer_grilla_easywave_ascii(grd_file)
    sub_arr, (nI, nJ), sub_bounds = leer_sshmax_subdom(sshmax_file)
    max_height_full, lat, lon = colocar_subdom_en_full(
        (nrows, ncols),
        (xmin, xmax, ymin, ymax),
        sub_arr,
        sub_bounds
    )

    eta_list = []
    time_list = []
    ssh_files = sorted(Path(run_out).glob("eWave.2D.*.ssh"))

    for f in ssh_files:
        try:
            t = int(f.stem.split(".")[2])  # tiempo en segundos
        except Exception:
            continue
        arr, (nI, nJ), sub_bounds_t = leer_sshmax_subdom(f)
        eta_full, _, _ = colocar_subdom_en_full(
            (nrows, ncols),
            (xmin, xmax, ymin, ymax),
            arr,
            sub_bounds_t
        )
        eta_list.append(eta_full)
        time_list.append(t)

    ds_vars = {
        "original_bathy": (["lat", "lon"], bathy),
        "deformed_bathy": (["lat", "lon"], bathy),
        "max_height":     (["lat", "lon"], max_height_full),
    }

    if eta_list:
        eta = np.stack(eta_list, axis=0)
        ds_vars["eta"] = (["time", "lat", "lon"], eta)

    ds = xr.Dataset(
        data_vars=ds_vars,
        coords={
            "lat": lat,
            "lon": lon,
            "time": time_list if eta_list else []
        },
        attrs={"source": "EasyWave outputs (sshmax + series eta)"}
    )

    ds.to_netcdf(nc_file)
    print(f"✅ NetCDF listo: {nc_file}")
    if eta_list:
        print(f"   Incluye {len(eta_list)} pasos de tiempo (eta)")
    else:
        print("⚠️ No se encontraron archivos .ssh (solo se incluyó sshmax)")

from datetime import timedelta

def generar_netcdf_eta(ssh_file, output_nc):
    """
    Convierte el archivo ASCII eWave.poi.ssh en un NetCDF con variable 'eta'
    de dimensiones (time, grid_npoints).
    El tiempo se almacena como strings en formato HH:MM:SS.
    """
    print(f"📈 Procesando mareogramas: {ssh_file}")

    # Leemos todas las líneas no vacías
    with open(ssh_file, "r") as f:
        lineas = [line.strip() for line in f if line.strip()]

    # Detectar y eliminar encabezado (líneas no numéricas)
    datos_numericos = []
    for line in lineas:
        partes = line.split()
        try:
            float(partes[0])  # si esto falla, no es una línea numérica
            datos_numericos.append(line)
        except ValueError:
            continue

    if not datos_numericos:
        print("⚠️ No se encontraron datos numéricos en el archivo.")
        return

    # Convertir a matriz numérica
    data = np.array([list(map(float, l.split())) for l in datos_numericos], dtype=np.float32)

    tiempo_min = data[:, 0]
    eta_vals = data[:, 1:]

    # Convertimos el tiempo a formato HH:MM:SS
    tiempo_hms = [
        str(timedelta(seconds=int(t * 60))) for t in tiempo_min
    ]

    # Crear dataset
    ds = xr.Dataset(
        data_vars={
            "eta": (["time", "grid_npoints"], eta_vals)
        },
        coords={
            "time": tiempo_hms,
            "grid_npoints": np.arange(eta_vals.shape[1])
        },
        attrs={
            "description": "Mareogramas simulados por EasyWave",
            "units": "meters",
            "time_format": "HH:MM:SS desde inicio de simulación"
        }
    )

    ds.to_netcdf(output_nc)
    print(f"✅ Archivo creado: {output_nc}")
    print(f"   Dimensiones: time={eta_vals.shape[0]}, grid_npoints={eta_vals.shape[1]}")
    print(f"   Ejemplo tiempo: {tiempo_hms[0]} → {tiempo_hms[-1]}")


# ================== MAIN ==================
grd_file = r"Datos/EasyWave/GrdASCII/grilla_ascii.grd"
flt_base = r"Datos/simulaciones-tsunami-hysea"
out_folder = r"Datos/EasyWave/Outputs"
poi_file = r"Datos/EasyWave/puntos.poi"
os.makedirs(out_folder, exist_ok=True)

flt_files = glob.glob(os.path.join(flt_base, "**", "*.flt"), recursive=True)

for flt_file in flt_files:
    rel_path = os.path.relpath(flt_file, flt_base)
    sim_name = os.path.splitext(rel_path.replace(os.sep, "_"))[0]

    run_out = os.path.join(out_folder, sim_name)
    os.makedirs(run_out, exist_ok=True)

    print(f"🌊 Procesando {sim_name} ...")

    subprocess.run([
        "easywave",
        "-grid", os.path.abspath(grd_file),
        "-source", os.path.abspath(flt_file),
        "-poi", os.path.abspath(poi_file),
        "-time", "120",
        "-gpu"                                       #SE AÑADE PARA QUE CORRA CON LA GPU
    ], cwd=run_out)

    subprocess.run(["sshmax2png.sh", "-grd", os.path.abspath(grd_file)], cwd=run_out)
    subprocess.run(["ssh2png.sh", "-grd", os.path.abspath(grd_file), "03600"], cwd=run_out)

    flt_dir = os.path.dirname(flt_file)
    file_heig = os.path.join(run_out, "eWave.2D.png")
    file_prop = os.path.join(run_out, "eWave.2D.03600.png")
    new_heig = os.path.join(flt_dir, "max_wave_heights.png")
    new_prop = os.path.join(flt_dir, "wave_propagation.png")

    if os.path.exists(file_heig):
        shutil.move(file_heig, new_heig)
    if os.path.exists(file_prop):
        shutil.move(file_prop, new_prop)

    # generar NetCDF con sshmax + series eta
    sshmax_file = os.path.join(run_out, "eWave.2D.sshmax")
    nc_file = os.path.join(flt_dir, "resultado_easywave.nc")
    if os.path.exists(sshmax_file):
        generar_netcdf(grd_file, sshmax_file, run_out, nc_file)

    # === Generar NetCDF con serie temporal eta ===
    ssh_file = os.path.join(run_out, "eWave.poi.ssh")
    nc_eta_file = os.path.join(flt_dir, "resultado_ts_easywave.nc")

    if os.path.exists(ssh_file):
        generar_netcdf_eta(ssh_file, nc_eta_file)
    else:
        print("⚠️ No se encontró el archivo eWave.poi.ssh (no se generó resultado_ts_easywave.nc)")


print("✅ Todos los procesos han finalizado correctamente.")


Procesando mw_8.1_02083_02083 ...


Error in file ewGpuNode.cu on line 58: no CUDA-capable device is detected



easyWave ver.2013-04-11
Cannot open file eWave.2D.sshmax
Cannot open file eWave.2D.03600.ssh
Procesando mw_8.3_03529_03529 ...

easyWave ver.2013-04-11


Error in file ewGpuNode.cu on line 58: no CUDA-capable device is detected


Cannot open file eWave.2D.sshmax
Cannot open file eWave.2D.03600.ssh
✅ Todos los procesos han finalizado.
